## Lesson Overview

**What this lesson teaches:** how to give Claude access to Anthropic's server-side web search tool and constrain its sources to trusted domains. Unlike Lessons 8 and 9, your Python code does not execute the tool locally: Anthropic runs the search and returns the results as part of the model response.

**What's happening under the hood:**
1. The setup cell loads `.env` and creates the client.
2. `web_search_schema` enables the server tool, limits it to five searches, and restricts results to `nih.gov`.
3. The prompt asks a current, evidence-based health question that benefits from search.
4. Claude decides whether to search, Anthropic performs it, and the final response includes source-backed content.
5. The inspection cells expose the response blocks so you can connect the readable answer to the underlying API structure.

The pattern to internalize: custom and client tools require your own execution loop; server tools are executed by Anthropic within a single API request. Your main responsibilities are configuring access, choosing source constraints, and inspecting the result.

# Lesson 10: Web search with trusted sources

This notebook builds directly on the tool-use ideas from Lessons 8 and 9. It uses Anthropic's web search tool to answer a question using only pages from the US National Institutes of Health.

Read each explanation before running the code below it. Pay particular attention to the difference between a **server tool** and a **local/client tool**, and to how `allowed_domains` changes what evidence Claude may use.

## How This Notebook Works

Lessons 8 and 9 followed a multi-step loop: Claude requested a tool, Python executed it, and Python sent a `tool_result` back. Web search is different because it is a **server-side tool**.

```text
+Lessons 8–9: your app → Claude requests tool → your Python runs tool → result returns to Claude
+Lesson 10:    your app → Anthropic runs search for Claude → final response returns to your app
+```

That is why this notebook can call `chat(...)` once instead of defining `run_tool`, `run_tools`, and `run_conversation`.

## Setup

The notebook looks for `.env` in either the project root or `Claude_API_Training/.env`. The expected values are:

```env
ANTHROPIC_API_KEY=...
MODEL_NAME=claude-sonnet-4-5
```

If no API key or package is available, the schema and local review cells still run; only the live search is skipped.

In [1]:
import os
from pathlib import Path

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

load_dotenv()
for env_path in (Path('.env'), Path('Claude_API_Training/.env')):
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

API_KEY = os.environ.get('ANTHROPIC_API_KEY')
model = os.environ.get('MODEL_NAME', 'claude-sonnet-4-5')
client = anthropic.Anthropic(api_key=API_KEY) if anthropic and API_KEY else None

print(f'Model: {model}')
print('Anthropic client ready.' if client else 'Live API demo unavailable; local review cells still work.')

Model: claude-haiku-4-5-20251001
Anthropic client ready.


## Message Helpers

These are the same basic helpers used in Lessons 8 and 9. Keeping them consistent makes it easier to compare notebooks. Notice that `stop_sequences` defaults to `None`, avoiding a shared mutable default list.

In [2]:
def add_user_message(messages, message):
    messages.append({
        'role': 'user',
        'content': message.content if hasattr(message, 'content') else message,
    })

def add_assistant_message(messages, message):
    messages.append({
        'role': 'assistant',
        'content': message.content if hasattr(message, 'content') else message,
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    if client is None:
        raise RuntimeError('Install anthropic and set ANTHROPIC_API_KEY before calling chat().')

    params = {
        'model': model,
        'max_tokens': 1500,
        'messages': messages,
        'temperature': temperature,
        'stop_sequences': stop_sequences or [],
    }
    if tools:
        params['tools'] = tools
    if system:
        params['system'] = system
    return client.messages.create(**params)

def text_from_message(message):
    return '\n'.join(
        block.text for block in message.content
        if getattr(block, 'type', None) == 'text'
    )

## Configure the Web Search Tool

This is a server-defined schema, so there is no custom `description` or `input_schema`.

- `type` selects the dated web-search tool version.
- `name` identifies the tool to Claude.
- `max_uses` caps search activity for this request, helping control cost and scope.
- `allowed_domains` acts as a source allowlist. Here Claude can search NIH pages, but not arbitrary websites.

Domain restrictions improve source control, but they do not guarantee that every claim is correct. You should still inspect citations and distinguish strong evidence from general guidance.

In [3]:
web_search_schema = {
    'type': 'web_search_20250305',
    'name': 'web_search',
    'max_uses': 5,
    'allowed_domains': ['nih.gov'],
}

web_search_schema

{'type': 'web_search_20250305',
 'name': 'web_search',
 'max_uses': 5,
 'allowed_domains': ['nih.gov']}

## Run a Source-Constrained Search

The prompt asks for evidence and practical guidance while explicitly avoiding a false one-size-fits-all conclusion. The schema supplies the real enforcement: search results must come from `nih.gov`.

Because Anthropic executes this server tool, one `chat` call can contain search activity and the final answer. No local tool-result loop is needed. Web search may incur additional API charges.

In [4]:
messages = []
add_user_message(
    messages,
    (
        'Using current NIH sources, explain which exercises and training principles '
        'best support leg-muscle growth for a healthy adult. Avoid claiming there is '
        'one universally best exercise. Summarize the evidence, give a practical '
        'beginner example, and clearly note important safety limitations.'
    ),
)

response = None
if client is None:
    print('Skipping live search. Configure the API key, then rerun this cell.')
else:
    response = chat(messages, tools=[web_search_schema])
    add_assistant_message(messages, response)
    print(text_from_message(response))

I'll search for current NIH sources on leg muscle growth and exercise principles.
Let me search for more specific safety and injury prevention guidance from NIH sources.
## Summary of Evidence from Current NIH Sources

### Key Training Principles for Leg Muscle Growth


Muscle hypertrophy is enhanced by higher volumes (≥10 sets/wk) and eccentric overload.
 However, research emphasizes that multiple approaches can work effectively. 
Hypertrophy appears less sensitive to qualitative differences in set structure once total volume, intensity and effort are adequately high.


**On exercise selection specifically:** 
Resistance training responses vary considerably depending on several training variables, with exercise selection being one such variable—different exercises have varying mechanical demands that can lead to differences in muscle growth, strength, and other related outcomes.
 This confirms there is no single "best" exercise.

**Load and repetition ranges:** 
For novice (untrained 

## Inspect the API Response

The rendered answer is useful, but the raw content blocks teach you how the API represents the result. Depending on the tool and SDK version, you may see text, server-tool activity, and citation information in separate or nested structures.

Compare each block's `type` with its data. This is the bridge between reading an answer in a notebook and building an application that processes sources programmatically.

In [5]:
if response is None:
    print('No live response to inspect yet.')
else:
    print('Stop reason:', response.stop_reason)
    print('Content block types:')
    for index, block in enumerate(response.content):
        block_type = getattr(block, 'type', type(block).__name__)
        print(f'  {index}: {block_type}')

    print('\nRaw response:')
    print(response.model_dump_json(indent=2))

Stop reason: end_turn
Content block types:
  0: text
  1: server_tool_use
  2: server_tool_use
  3: web_search_tool_result
  4: web_search_tool_result
  5: text
  6: server_tool_use
  7: server_tool_use
  8: web_search_tool_result
  9: web_search_tool_result
  10: text
  11: text
  12: text
  13: text
  14: text
  15: text
  16: text
  17: text
  18: text
  19: text
  20: text
  21: text
  22: text
  23: text
  24: text
  25: text
  26: text
  27: text
  28: text
  29: text
  30: text
  31: text
  32: text
  33: text
  34: text
  35: text
  36: text

Raw response:
{
  "id": "msg_011Cdq8yaiXBnF6DkAS3aKTA",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "I'll search for current NIH sources on leg muscle growth and exercise principles.",
      "type": "text"
    },
    {
      "id": "srvtoolu_01E1wsyzzaXs8oVSFNugbUZT",
      "caller": null,
      "input": {
        "query": "NIH leg muscle growth exercises resistance training"
      },
      "name": "web_

## Local Sanity Checks

These checks require no API call. They confirm the configuration you are about to send and catch simple mistakes such as the wrong tool name or an overly broad domain list.

In [6]:
assert web_search_schema['name'] == 'web_search'
assert web_search_schema['max_uses'] == 5
assert web_search_schema['allowed_domains'] == ['nih.gov']

print('Tool version:', web_search_schema['type'])
print('Search limit:', web_search_schema['max_uses'])
print('Allowed sources:', ', '.join(web_search_schema['allowed_domains']))

Tool version: web_search_20250305
Search limit: 5
Allowed sources: nih.gov


## Practice: Learn by Changing One Constraint

Try these one at a time, predict the effect before running, and then inspect the raw response:

1. **Prompt change:** ask for evidence about progressive resistance training instead of a broad exercise recommendation.
2. **Domain change:** use another trusted domain appropriate to your question. Do not merely remove the allowlist—notice how explicit source boundaries affect reliability.
3. **Budget change:** reduce `max_uses` to `1`. Does the answer become narrower or less well supported?
4. **No-search comparison:** call `chat(messages)` without tools and compare freshness, citations, and confidence.

Reflection questions:
- Who executes this tool: your notebook or Anthropic?
- Which settings control search scope and source scope?
- Why is a trusted-domain answer still not a substitute for professional medical advice?

## Summary

- Web search is a **server-side tool**, so Anthropic executes it inside the API request.
- `allowed_domains` constrains which sites may supply search results.
- `max_uses` limits how many searches Claude may perform for the request.
- A single response can contain both tool activity and the final source-backed answer.
- Inspecting raw content blocks is essential when you want to handle citations or tool activity in an application.
- Source restrictions improve evidence quality, but you must still evaluate claims and communicate uncertainty.

The key contrast with Lessons 8 and 9 is ownership of execution: local tools need your dispatcher and tool-result loop; server tools need careful configuration and response handling.

## New Use Case: Find the Newly Appointed Federal Reserve Chair's Biography

This example adapts the same web search tool to a current-events question. The domain allowlist limits results to the Federal Reserve's official website, so the answer is based on a primary government source rather than news reports or social media.

In [7]:
fed_web_search_schema = {
    'type': 'web_search_20250305',
    'name': 'web_search',
    'max_uses': 5,
    'allowed_domains': ['federalreserve.gov'],
}

fed_web_search_schema

{'type': 'web_search_20250305',
 'name': 'web_search',
 'max_uses': 5,
 'allowed_domains': ['federalreserve.gov']}

### Search the Official Federal Reserve Website

The prompt asks Claude to verify who was most recently appointed rather than assuming a name. It then requests a concise biography and citations to the official Federal Reserve pages used.

In [8]:
fed_messages = []
add_user_message(
    fed_messages,
    (
        'Using only the official Federal Reserve website, identify the person most '
        'recently appointed as Chair of the Board of Governors of the Federal Reserve '
        'System. Then provide a concise biography covering his education, major career '
        'roles, previous Federal Reserve or government service, and appointment date. '
        'Clearly distinguish the appointment or nomination date from the date he took '
        'office, if applicable. Cite the official Federal Reserve pages supporting the '
        'answer, and say explicitly if the official site does not yet confirm the appointment.'
    ),
)

fed_response = None
if client is None:
    print('Skipping live search. Configure the API key, then rerun this cell.')
else:
    fed_response = chat(fed_messages, tools=[fed_web_search_schema])
    add_assistant_message(fed_messages, fed_response)
    print(text_from_message(fed_response))

I'll search for information about the current Federal Reserve Chair from the official Federal Reserve website.
Now I need to get Kevin Warsh's full biography from the Federal Reserve website.
Based on my search of the official Federal Reserve website, I can now provide you with the information:

## Most Recently Appointed Chair


Kevin Warsh took office as chairman of the Board of Governors of the Federal Reserve System on May 22, 2026, for a four-year term ending on May 21, 2030.


## Appointment/Confirmation Dates

- **Nomination Date:** 
President Donald J. Trump nominated Mr. Warsh on March 4, 2026.

- **Senate Confirmation Dates:** 
He was confirmed by the United States Senate to serve as a member of the Board on May 12, and as chairman of the Board on May 13.

- **Oath of Office Date:** 
Kevin Warsh took the oath of office as chairman and a member of the Board of Governors of the Federal Reserve System on May 22, 2026.


## Biography

**Education:** 
He received an AB from Stanfo